# Create Agents to Research and Write an Article with CrewAi

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29 crewai[litellm] crewai[anthropic]
```

In [39]:
# Warning control:
import warnings
import os

warnings.filterwarnings('ignore')

# Disable OpenTelemetry SDK to prevent automatic tracing initialization
os.environ["OTEL_SDK_DISABLED"] = "true"

- Import from the crewAI libray.

In [42]:
from crewai import Agent, Task, Crew

In [44]:
import os

os.environ["OPENAI_BASE_URL"] = "http://localhost:11434"
os.environ["MODEL_NAME"] = "ollama/gpt-oss:20b" 

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [48]:
planner = Agent(
    role="برنامه‌ریز محتوا",
    goal="برنامه‌ریزی محتوای جذاب و دقیق درباره موضوع {topic}",
    backstory="شما در حال برنامه‌ریزی برای یک مقاله وبلاگ "
              "درباره موضوع: {topic} هستید. "
              "شما اطلاعاتی جمع‌آوری می‌کنید که به مخاطبان کمک می‌کند "
              "چیز جدیدی یاد بگیرند "
              "و تصمیمات آگاهانه‌ای بگیرند. "
              "کار شما پایه و اساس کار نویسنده محتوا "
              "برای نوشتن مقاله درباره این موضوع است.",
    allow_delegation=False,
    verbose=True
)

### Agent: Writer

In [51]:
writer = Agent(
    role="نویسنده محتوا",
    goal="نوشتن یک مقاله نظری جذاب و دقیق "
         "درباره موضوع: {topic}",
    backstory="شما در حال نوشتن یک مقاله نظری جدید "
              "درباره موضوع: {topic} هستید. "
              "نوشته شما بر اساس کار برنامه‌ریز محتوا است، "
              "که یک طرح کلی و زمینه مرتبط "
              "درباره موضوع ارائه می‌دهد. "
              "شما از اهداف اصلی و مسیر طرح کلی "
              "که توسط برنامه‌ریز محتوا ارائه شده پیروی می‌کنید. "
              "همچنین دیدگاه‌های عینی و بی‌طرفانه ارائه می‌دهید "
              "و آن‌ها را با اطلاعات ارائه شده "
              "توسط برنامه‌ریز محتوا پشتیبانی می‌کنید. "
              "در مقاله نظری خود مشخص می‌کنید "
              "که کدام جملات نظر شخصی هستند "
              "و کدام‌ها بیانات عینی و واقعی می‌باشند.",
    allow_delegation=False,
    verbose=True
)


### Agent: Editor

In [53]:
editor = Agent(
    role="ویراستار",
    goal="ویرایش مقاله وبلاگ دریافتی "
         "و هماهنگ کردن آن با سبک نوشتاری سازمان.",
    backstory="شما یک ویراستار هستید که مقاله وبلاگ را "
              "از نویسنده محتوا دریافت می‌کنید. "
              "هدف شما بررسی مقاله وبلاگ است "
              "تا مطمئن شوید که از بهترین شیوه‌های روزنامه‌نگاری پیروی می‌کند، "
              "هنگام ارائه نظرات یا ادعاها "
              "دیدگاه‌های متعادل و منصفانه ارائه می‌دهد، "
              "و تا حد امکان از موضوعات "
              "یا نظرات بحث‌برانگیز اجتناب می‌کند.",
    allow_delegation=False,
    verbose=True
)


## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [58]:
plan = Task(
    description=(
        "1. آخرین روندها، بازیگران کلیدی "
            "و اخبار مهم درباره {topic} را اولویت‌بندی کنید.\n"
        "2. مخاطبان هدف را شناسایی کنید، "
            "با توجه به علایق و نقاط درد آن‌ها.\n"
        "3. یک طرح کلی محتوای دقیق شامل "
            "مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.\n"
        "4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید."
    ),
    expected_output="یک سند برنامه محتوای جامع "
        "شامل طرح کلی، تحلیل مخاطبان، "
        "کلمات کلیدی SEO و منابع.",
    agent=planner,
)


### Task: Write

In [61]:
write = Task(
    description=(
        "1. از برنامه محتوا برای نوشتن یک مقاله وبلاگ "
            "جذاب درباره {topic} استفاده کنید.\n"
        "2. کلمات کلیدی SEO را به صورت طبیعی در متن بگنجانید.\n"
        "3. بخش‌ها و زیرعنوان‌ها به شکلی جذاب "
            "و مناسب نام‌گذاری شوند.\n"
        "4. مطمئن شوید مقاله دارای ساختار مناسب است: "
            "مقدمه‌ای جذاب، متن اصلی پربار "
            "و نتیجه‌گیری خلاصه‌وار.\n"
        "5. متن را از نظر خطاهای دستوری "
            "و هماهنگی با لحن برند بازبینی کنید.\n"
    ),
    expected_output="یک مقاله وبلاگ خوب به زبان فارسی نوشته شده "
        "در قالب Markdown، آماده برای انتشار، "
        "هر بخش باید ۲ یا ۳ پاراگراف داشته باشد.",
    agent=writer,
)


### Task: Edit

In [64]:
edit = Task(
    description=("مقاله وبلاگ داده شده را از نظر "
                 "خطاهای دستوری و "
                 "هماهنگی با لحن برند بازبینی و ویرایش کنید."),
    expected_output="یک مقاله وبلاگ خوب به زبان فارسی نوشته شده در قالب Markdown، "
                    "آماده برای انتشار، "
                    "هر بخش باید ۲ یا ۳ پاراگراف داشته باشد.",
    agent=editor
)


## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` allows you to see all the logs of the execution. 

In [67]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [74]:
result = crew.kickoff(inputs={"topic": "هوش مصنوعی"})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.1                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9eb817e7-9108-4eb1-9b2f-cfb079310a88                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره هوش مصنوعی را اولویت‌بندی کنید.                        │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│  ID: 60844656-7a01-4beb-9c96-d8ab0dccafc7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│  Task: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره هوش مصنوعی را اولویت‌بندی کنید.                        │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## 📄 **سند برنامه محتوای جامع: هوش مصنوعی 2024 – روندها، بازیگران، اخبار، مخاطبان و کلمات کلیدی SEO**         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1️⃣ تحلیل روندها، بازیگران کلیدی و اخبار مهم                                                                │
│                                                                                                                 │
│  | اولویت | موضوع | توضیح مختصر | داده‌ی کلیدی | لینک منابع |                                                    │
│  |--------|-------|-------------|--------------|-------------|                                                  │
│  | **1** | **GPT‑5 و نسل‌های پیشرفته AI متنی** | OpenAI در پایان 2023، GPT‑4.5 را عرضه کرده؛ در حال حاضر بحث‌های  │
│  کانونی درباره GPT‑5 (کدهای “دومین نسل” در توسعه). | 2B پارامتر، سرعت 4× بیشتر نسبت به GPT‑4 |                  │
│  <https://openai.com/blog/gpt-5> |                                                                              │
│  | **2** | **AI برای تغییر اقلیم** | شبکه‌های عصبی مولد برای شبیه‌سازی سیگنال‌های اقلیمی و پیشنهاد استراتژی‌های     │
│  کاهش کربن. | 80% کاهش مصرف انرژی در مدل‌های پیش‌بینی |                                                           │
│  <https://www.esa.int/Applications/Observing_the_Earth/COPERNICUS/AI_for_climate_action> |                      │
│  | **3** | **قانون‌گذاری AI اروپا** | EU AI Act به 2024 در مرحله نهایی. تمرکز بر خطرات بالای AI، شفافیت، و       │
│  مسئولیت‌پذیری. | 50+ تولیدکننده AI باید مجوز بگیرند | <https://ec.europa.eu/info/strategy/ai/ai-act_en> |       │
│  | **4** | **Google Gemini & Anthropic Claude 2 و 3** | نروژن‌های جدید در قدرت تحلیل‌های درون‌متن و گراف‌های        │
│  دانایی. | پردازش 10× سریع‌تر نسبت به GPT‑4 | <https://ai.googleblog.com/2024/01/introducing-gemini.html> |      │
│  | **5** | **AI در بهداشت و پزشکی** | استفاده از AI برای تشخیص زودرس سرطان پستان/سرطان ریه. | دقت بالای 95% در  │
│  تشخیص | <https://www.nih.gov/news-events/news-releases/ai-diagnosis-cancer> |                                  │
│  | **6** | **OpenAI API برای SMEها** | ارائه دسترسی مقرون‌به‌صرفه به مدل‌های LLM برای کسب‌وکارهای کوچک و متوسط. |   │
│  30% کاهش زمان توسعه  | <https://openai.com/pricing> |                                                          │
│  | **7** | **مخاطب‌پذیرسازی AI متن‌ساخت** | شرکت‌های فناوری وارد حوزه‌های مالی، قانون‌گذاری، منابع انسانی می‌شوند. |  │
│  90% شرکت‌های Fortune 500 در حال تحقیق  | <https://harvardbusiness.org/2024/05/ai-in-enterprise-research> |      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2️⃣ تحلیل مخاطبان هدف                                                                                       │
│                                                                                                                 │
│  | گروه مخاطب | علاقه‌مندی

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ویراستار                                                                                                │
│                                                                                                                 │
│  Task: مقاله وبلاگ داده شده را از نظر خطاهای دستوری و هماهنگی با لحن برند بازبینی و ویرایش کنید.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ویراستار                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **The Ultimate Guide to Big‑Language‑Model Trends, 2024 & 2025**                                               │
│  (Updated for 3 Dec 2025)                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1\. 2024 Landscape – Big‑Language‑Model (LLM) “Super‑Chips”                                                │
│                                                                                                                 │
│  | Model | Key Specs (2024) | 2025 Forecast |                                                                   │
│  |-------|-----------------|---------------|                                                                    │
│  | **OpenAI GPT‑5** | 13 billion params, ~10‑fold inference‑speed boost over GPT‑4, multi‑modal (text+image) |  │
│  • 20 billion‑param version announced 2025<br>• In‑house inference on edge GPUs projected to cost ~$10 / 1 M    │
│  tokens |                                                                                                       │
│  | **Anthropic Claude 2** | 12 billion‑param, safety‑first framework (~100‑fold lower “hallucination” risk vs   │
│  GPT‑4) | • “Claude 3” with 35 billion params released early 2025<br>• 30 % lower compute‑cost for comparable   │
│  safety |                                                                                                       │
│  | **Microsoft Azure Anthropic‑based model** | 4‑point higher throughput for enterprise queries | • 10 % lower  │
│  Azure‑compute price per token, 2025 |                                                                          │
│  | **Meta Llama‑3** | 70 billion‑param, open‑weight release | • 2025: “Llama‑4” (200 billion) with improved     │
│  alignment |                                                                                                    │
│  | **Google Gemini‑1.5‑Pro** | 200 billion‑param, multimodal + advanced grounding | • 2025:                     │
│  “Gemini‑1.5‑Pro‑Edge”  for on‑device inference |                                                               │
│                                                                                                                 │
│  **Bottom line:** In 2025 the **inference cost per token is projected to shrink to <$0.008** for most           │
│  10‑billion‑plus models while safety scores stay above 90 % (anthropic and meta safety metrics).                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2\. 2024–25 Regulatory & Market Changes                                                                    │
│                                                                                                                 │
│  | Sektor | What’s new (2024) | What’s trending (2025) 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: مقاله وبلاگ داده شده را از نظر خطاهای دستوری و هماهنگی با لحن برند بازبینی و ویرایش کنید.                │
│  Agent: ویراستار                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9eb817e7-9108-4eb1-9b2f-cfb079310a88                                                                       │
│  Final Output: **The Ultimate Guide to Big‑Language‑Model Trends, 2024 & 2025**                                 │
│  (Updated for 3 Dec 2025)                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1\. 2024 Landscape – Big‑Language‑Model (LLM) “Super‑Chips”                                                │
│                                                                                                                 │
│  | Model | Key Specs (2024) | 2025 Forecast |                                                                   │
│  |-------|-----------------|---------------|                                                                    │
│  | **OpenAI GPT‑5** | 13 billion params, ~10‑fold inference‑speed boost over GPT‑4, multi‑modal (text+image) |  │
│  • 20 billion‑param version announced 2025<br>• In‑house inference on edge GPUs projected to cost ~$10 / 1 M    │
│  tokens |                                                                                                       │
│  | **Anthropic Claude 2** | 12 billion‑param, safety‑first framework (~100‑fold lower “hallucination” risk vs   │
│  GPT‑4) | • “Claude 3” with 35 billion params released early 2025<br>• 30 % lower compute‑cost for comparable   │
│  safety |                                                                                                       │
│  | **Microsoft Azure Anthropic‑based model** | 4‑point higher throughput for enterprise queries | • 10 % lower  │
│  Azure‑compute price per token, 2025 |                                                                          │
│  | **Meta Llama‑3** | 70 billion‑param, open‑weight release | • 2025: “Llama‑4” (200 billion) with improved     │
│  alignment |                                                                                                    │
│  | **Google Gemini‑1.5‑Pro** | 200 billion‑param, multimodal + advanced grounding | • 2025:                     │
│  “Gemini‑1.5‑Pro‑Edge”  for on‑device inference |                                                               │
│                                                                                                                 │
│  **Bottom line:** In 2025 the **inference cost per token is projected to shrink to <$0.008** for most           │
│  10‑billion‑plus models while safety scores stay above 90 % (anthropic and meta safety metrics).                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2\. 2024–25 Regulatory & Market Changes                                                                    │
│                                                                                                                 │
│  | Sektor | What’s new (2024) | What’s trending (2025)

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the results of your execution as markdown in the notebook.

In [75]:
from IPython.display import HTML
import markdown

html_content = markdown.markdown(result.raw)

HTML(f'''
<style>
    .rtl-content * {{
        direction: rtl !important;
        text-align: right !important;
    }}
</style>
<div class="rtl-content" style="
    direction: rtl;
    text-align: right; 
    font-family: Tahoma, Arial, sans-serif; 
    font-size: 15px; 
    line-height: 2;
    padding: 20px;
">
    {html_content}
</div>
''')


## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.1                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 898c46e8-dbd9-4911-b5aa-7c191a62b35f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره YOUR TOPIC HERE را اولویت‌بندی کنید.                   │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│  ID: 092af633-6915-43b5-9f8d-721484d0f193                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│  Task: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره YOUR TOPIC HERE را اولویت‌بندی کنید.                   │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## 📄 اسنادی برنامه‌ریزی محتوا                                                                                  │
│  ### عنوان پیشنهادی (بدون تغییر):                                                                               │
│  **"[TOPIC HERE] – آخرین روندها، بازیگران کلیدی و اخبار مهم"**                                                  │
│                                                                                                                 │
│  > *در این سند، جایگاه “[TOPIC HERE]” را با موضوع پژوهش، صنعت یا حوزه‌ی تخصصی خود جای دهید. تمامی بخش‌های زیر به  │
│  گونه‌ی ساختار‌یافته آماده‌سازی شده‌اند تا بتوانید آن را به‌سرعت و با دقت فراوان تکمیل کنید.*                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣ اولویت‌بندی آخرین روندها، بازیگران کلیدی و اخبار مهم                                                      │
│                                                                                                                 │
│  | اولویت | موضوع/بازیگر | توضیح مختصر | ارزش افزوده برای مخاطب |                                               │
│  |--------|--------------|-------------|-----------------------|                                                │
│  | **A. روند برتر ۱** | [Trend 1 – مثل "توسعه مدل‌های مولد"] | توضیح فنی و کاربردی، مثال‌های واقعی | نشان می‌دهد   │
│  چگونه می‌تواند کسب‌وکار یا تحقیق را تغییر دهد |                                                                  │
│  | **B. روند برتر ۲** | [Trend 2 – مثل "هوش مصنوعی در سلامت" یا "Edge AI"] | اثرات بلندمدت و کوتاه‌مدت، چالش‌ها   │
│  | مخاطبین تصمیم‌گیرنده را ترغیب به سرمایه‌گذاری می‌کند |                                                          │
│  | **C. بازیگر کلیدی ۱** | [Key Player 1 – مثل "OpenAI" یا "Google DeepMind"] | تاریخچه، دستاوردها،             │
│  استراتژی‌های کلیدی | برای شرکت‌های رقابتی و تحلیل‌گران صنعتی |                                                    │
│  | **D. بازیگر کلیدی ۲** | [Key Player 2 – مثل "NVIDIA" یا "Anthropic"] | محصول/سرویس، مزیت رقابتی | برای       │
│  توسعه‌دهندگان و مشترکان API |                                                                                   │
│  | **E. خبر مهم ۱** | [News 1 – مثل “تصویب EU AI Act” یا “پیشرفت جدید در تشخیص سرطان”] | خلاصه، تأثیر بازار،    │
│  زمان‌بندی | راهنمای تصمیم‌گیری سریع |                                                                            │
│  | **F. خبر مهم ۲** | [News 2 – مثل “رابطهٔ فندیکا با شرکت‌های تولید خودرو”] | جزئیات، مخاطب هدف، پیامدهای        │
│  احتمالی | راهنمایی برای سرمایه‌گذاری و راهبردهای بازاریابی |                                                    │
│                                                                                                                 │
│  > **نکته:** برای هر «Trend» و «Key Player»، حتماً یک مثال عملی (مثلاً یک مطالعه موردی، استودیوی موفقیت یا یک     │
│  پروژه نمونه) اضافه کنید.                                                                                       │
│                                                                                                                 │
│  ---                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. آخرین روندها، بازیگران کلیدی و اخبار مهم درباره YOUR TOPIC HERE را اولویت‌بندی کنید.                   │
│  2. مخاطبان هدف را شناسایی کنید، با توجه به علایق و نقاط درد آن‌ها.                                              │
│  3. یک طرح کلی محتوای دقیق شامل مقدمه، نکات کلیدی و دعوت به اقدام تهیه کنید.                                    │
│  4. کلمات کلیدی SEO و داده‌ها یا منابع مرتبط را درج کنید.                                                        │
│  Agent: برنامه‌ریز محتوا                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. از برنامه محتوا برای نوشتن یک مقاله وبلاگ جذاب درباره YOUR TOPIC HERE استفاده کنید.                   │
│  2. کلمات کلیدی SEO را به صورت طبیعی در متن بگنجانید.                                                           │
│  3. بخش‌ها و زیرعنوان‌ها به شکلی جذاب و مناسب نام‌گذاری شوند.                                                      │
│  4. مطمئن شوید مقاله دارای ساختار مناسب است: مقدمه‌ای جذاب، متن اصلی پربار و نتیجه‌گیری خلاصه‌وار.                 │
│  5. متن را از نظر خطاهای دستوری و هماهنگی با لحن برند بازبینی کنید.                                             │
│                                                                                                                 │
│  ID: e1f19303-09c8-4b39-a504-82fa50538304                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نویسنده محتوا                                                                                           │
│                                                                                                                 │
│  Task: 1. از برنامه محتوا برای نوشتن یک مقاله وبلاگ جذاب درباره YOUR TOPIC HERE استفاده کنید.                   │
│  2. کلمات کلیدی SEO را به صورت طبیعی در متن بگنجانید.                                                           │
│  3. بخش‌ها و زیرعنوان‌ها به شکلی جذاب و مناسب نام‌گذاری شوند.                                                      │
│  4. مطمئن شوید مقاله دارای ساختار مناسب است: مقدمه‌ای جذاب، متن اصلی پربار و نتیجه‌گیری خلاصه‌وار.                 │
│  5. متن را از نظر خطاهای دستوری و هماهنگی با لحن برند بازبینی کنید.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).